# Face Mask Detection — VGG16 Transfer Learning

Same problem as notebook 1, but this time using a pretrained VGG16 instead of building a CNN from scratch.  
The idea: freeze VGG16's layers and only train a single Dense layer on top.

**Why?** Our dataset is tiny (~1,376 images). VGG16 was trained on 14M images — we can borrow those learned features.

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## 2. Load the Dataset

Same dataset as notebook 1 — two folders, resize to 224x224.

In [ ]:
# Clone the dataset (run once)
# !git clone https://github.com/ricklon/pyimagesearch-face-mask-detector.git

dataset_path = "/content/pyimagesearch-face-mask-detector/dataset"
categories = ['without_mask', 'with_mask']   # 0 = no mask, 1 = mask

print("Classes:", os.listdir(dataset_path))

In [ ]:
data = []
for category in categories:
    path = os.path.join(dataset_path, category)
    label = categories.index(category)

    for file in os.listdir(path):
        img_path = os.path.join(path, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))
        data.append([img, label])

print(f"Total images loaded: {len(data)}")

## 3. Preprocess & Split

Important difference from notebook 1: here we use `preprocess_input` instead of `/255`.  
VGG16 expects its specific preprocessing (subtracts ImageNet channel means) — using `/255` here would give bad results because the pretrained weights expect a different input distribution.

In [ ]:
random.shuffle(data)

X = np.array([item[0] for item in data])
y = np.array([item[1] for item in data])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class distribution: no_mask={sum(y==0)}, mask={sum(y==1)}")

In [ ]:
# VGG16 preprocessing — NOT /255
X = preprocess_input(X)

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 4. Build the Model

Take VGG16, chop off the last layer (1000-class softmax), freeze everything, add our own Dense(1).  
We end up with 134M frozen params and only 4,097 trainable params — that's the whole point of transfer learning.

In [ ]:
vgg = VGG16()

# copy all layers except the final classification layer
model = Sequential()
for layer in vgg.layers[:-1]:
    model.add(layer)

# freeze everything
for layer in model.layers:
    layer.trainable = False

# our classification head
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Train

Data augmentation + EarlyStopping + ModelCheckpoint.  
Since we're only training 4K params, this is very fast even without heavy augmentation.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2
)

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

## 6. Evaluate

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred,
      target_names=['without_mask', 'with_mask']))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Takeaways

- VGG16 transfer learning hits ~96% accuracy vs ~90-93% from scratch (notebook 1)
- We only trained 4,097 parameters — the rest came free from ImageNet pretraining
- `preprocess_input` is not the same as `/255` — using the wrong one tanks accuracy
- Freezing all layers is the safe bet for tiny datasets (<2K images)
- EarlyStopping stopped training around epoch 5-7 — the model converges fast with frozen features
- **Bottom line:** Transfer learning >>> training from scratch when you have limited data